# 01 Data Loading and EDA

This notebook loads the processed GSE40732 expression and metadata CSV files created by `src/extract_gse40732.R`. It performs lightweight exploratory checks only: no model training is done here.

## 1. Setup

Import the core Python packages and define project paths.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "figures"

expression_path = processed_dir / "gse40732_expression.csv"
metadata_path = processed_dir / "gse40732_metadata.csv"
labels_path = processed_dir / "gse40732_labels.csv"

class_plot_path = figures_dir / "gse40732_class_distribution.png"
expression_plot_path = figures_dir / "gse40732_expression_distribution.png"

figures_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

## 2. Load Processed CSV Files

The expression matrix is expected to have genes or probes as rows and samples as columns. The metadata table is expected to have one row per sample.

In [ ]:
for path in [expression_path, metadata_path]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}. Run src/extract_gse40732.R first."
        )

expression = pd.read_csv(expression_path, index_col=0)
metadata = pd.read_csv(metadata_path, index_col=0)

expression = expression.apply(pd.to_numeric, errors="coerce")

print(f"Expression matrix shape: {expression.shape}")
print(f"Metadata table shape: {metadata.shape}")

## 3. Inspect Metadata Columns

Before creating labels, inspect available metadata columns. The label-cleaning code checks common GEO fields and prints examples if automatic labeling fails.

In [ ]:
print("Metadata columns:")
for column in metadata.columns:
    print(f"- {column}")

candidate_columns = [
    "characteristics_ch1",
    "characteristics_ch1.1",
    "title",
    "source_name_ch1",
]

available_candidates = [col for col in candidate_columns if col in metadata.columns]
print("\nCandidate label columns found:", available_candidates)

if available_candidates:
    display(metadata[available_candidates].head(10))
else:
    print("No common candidate label columns found. Showing metadata preview:")
    display(metadata.head(10))

## 4. Create Asthma/Control Labels

Labels are inferred from common GEO metadata fields. The rules are intentionally simple and transparent so they can be reviewed before modeling.

In [ ]:
def infer_asthma_label(row):
    text_parts = []
    for col in available_candidates:
        value = row.get(col)
        if pd.notna(value):
            text_parts.append(str(value))

    text = " | ".join(text_parts).lower()

    control_terms = [
        "asthma: false",
        "non-asthma",
        "non asthma",
        "non-asthmatic",
        "non asthmatic",
        "control",
        "healthy",
        "normal",
    ]
    asthma_terms = [
        "asthma: true",
        "asthmatic",
        "asthma",
        "severe asthma",
    ]

    if any(term in text for term in control_terms):
        return "Control"
    if any(term in text for term in asthma_terms):
        return "Asthma"
    return np.nan


metadata_aligned = metadata.reindex(expression.columns)

if metadata_aligned.isna().all(axis=None):
    print("Metadata index did not align cleanly to expression columns.")
    print("First expression sample IDs:", list(expression.columns[:10]))
    print("First metadata row IDs:", list(metadata.index[:10]))

metadata_aligned["label"] = metadata_aligned.apply(infer_asthma_label, axis=1)

label_counts = metadata_aligned["label"].value_counts(dropna=False)
print(label_counts)

unlabeled = metadata_aligned[metadata_aligned["label"].isna()]
if not unlabeled.empty:
    print("\nCould not automatically label these samples. Metadata examples:")
    preview_cols = available_candidates if available_candidates else metadata.columns[:8].tolist()
    display(unlabeled[preview_cols].head(15))

labels = metadata_aligned[["label"]].dropna().copy()
labels.index.name = "sample_id"

if labels.empty:
    raise ValueError(
        "No labels could be created automatically. Inspect the metadata examples above "
        "and update the label-cleaning rules."
    )

labels.to_csv(labels_path)
print(f"Saved labels to {labels_path}")
display(labels.head())

## 5. Check Missing Expression Values

Check whether the numeric expression matrix contains missing values after CSV loading.

In [ ]:
missing_total = int(expression.isna().sum().sum())
missing_by_sample = expression.isna().sum(axis=0).sort_values(ascending=False)
missing_by_feature = expression.isna().sum(axis=1).sort_values(ascending=False)

print(f"Total missing expression values: {missing_total}")
print("\nSamples with the most missing values:")
display(missing_by_sample.head(10).to_frame("missing_values"))
print("\nFeatures with the most missing values:")
display(missing_by_feature.head(10).to_frame("missing_values"))

## 6. Plot Class Distribution

Visualize the number of asthma and control samples identified from metadata.

In [ ]:
class_counts = labels["label"].value_counts().reindex(["Control", "Asthma"]).dropna()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(class_counts.index, class_counts.values, color=["#4C78A8", "#F58518"])
ax.set_title("GSE40732 Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Number of samples")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

for index, value in enumerate(class_counts.values):
    ax.text(index, value, int(value), ha="center", va="bottom")

fig.tight_layout()
fig.savefig(class_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved class distribution figure to {class_plot_path}")

## 7. Plot Expression Value Distribution

Plot the overall distribution of expression values. For large matrices, a reproducible random sample of values is used for plotting.

In [ ]:
values = expression.to_numpy(dtype=float).ravel()
values = values[np.isfinite(values)]

max_plot_values = 1_000_000
if values.size > max_plot_values:
    rng = np.random.default_rng(3888)
    values_to_plot = rng.choice(values, size=max_plot_values, replace=False)
else:
    values_to_plot = values

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(values_to_plot, bins=60, color="#54A24B", edgecolor="white")
ax.set_title("GSE40732 Expression Value Distribution")
ax.set_xlabel("Expression value")
ax.set_ylabel("Frequency")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

fig.tight_layout()
fig.savefig(expression_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Plotted {values_to_plot.size:,} expression values")
print(f"Saved expression distribution figure to {expression_plot_path}")